# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

  - 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies

We'll first install all our required libraries.

> NOTE: If you're running this locally - please skip this step.

In [5]:
!pip install -qU langchain langchain_openai langchain-community langgraph arxiv

## Task 2: Environment Variables

We'll want to set both our OpenAI API key and our LangSmith environment variables.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE5 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/tools) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/tools/arxiv)

####🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [6]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?
LSY ANSWER: 	
We add our selected tools to the tool_belt, then bind tool_belt to the model; in this case, OpenAI's GPT-4o.
GPT-4o (or other function-calling models) decides when to invoke a tool based on the prompt and context. The LangChain framework executes these tool calls as needed.

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [8]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [9]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [10]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [11]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [12]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [13]:
compiled_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?    
LSY ANSWER: 
No, there is no built-in limit to the number of cycles in LangGraph. To impose a limit, we can add a conditional edge that checks the number of iterations in the state. If it exceeds a set threshold, the graph returns the END node. This can be done by tracking a cycle counter and exiting once the limit is reached, preventing infinite loops.


## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [14]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="Who is the current captain of the Winnipeg Jets?")]}

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_vh327nC0BCLZffdF7vLS34hp', 'function': {'arguments': '{"query":"current captain of the Winnipeg Jets 2023"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 162, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4691090a87', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-130a4f47-87a3-4637-974d-7f7c0277f489-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current captain of the Winnipeg Jets 2023'}, 'id': 'call_vh327nC0BCLZffdF7vLS34hp', 'type': 'tool_call'}], usage_metadata={'input_tokens': 162, 'output_t

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [15]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!")]}

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_vHMu3JXG5IBxTYcCTh2lvr6s', 'function': {'arguments': '{"query":"QLoRA"}', 'name': 'arxiv'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 178, 'total_tokens': 195, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4691090a87', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-7c743316-fec9-4e3a-9184-c00a3e14f21f-0', tool_calls=[{'name': 'arxiv', 'args': {'query': 'QLoRA'}, 'id': 'call_vHMu3JXG5IBxTYcCTh2lvr6s', 'type': 'tool_call'}], usage_metadata={'input_tokens': 178, 'output_tokens': 17, 'total_tokens': 195, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'a

####🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

LSY ANSWER: 

The agent successfully broke down the request, used the correct tools, and formatted the answer clearly.

User Query Processing
User asks: “Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!”
Agent determines it needs to:
•	Search Arxiv for QLoRA
•	Extract author names
•	Search for each author’s latest tweet

Query Arxiv for the QLoRA Paper
Calls arxiv tool with:
{“query”: “QLoRA”}
Receives paper details with authors:
[“Tim Dettmers”, “Artidoro Pagnoni”, “Ari Holtzman”, “Luke Zettlemoyer”]

Extract Authors from Arxiv Response
Parses authors for the next step

Query Tavily for Each Author’s Latest Tweet
Calls tavily_search_results_json tool four times:
{“query”: “Tim Dettmers latest tweet”}
{“query”: “Artidoro Pagnoni latest tweet”}
{“query”: “Ari Holtzman latest tweet”}
{“query”: “Luke Zettlemoyer latest tweet”}
Receives tweet URLs and content

Retrieve and Format Final Response
Formats structured output:
	1.	Tim Dettmers:
	Latest Tweet: 			https://twitter.com/Tim_Dettmers/status/1683118705956491264

	2.	Artidoro Pagnoni:
	Latest Tweet: 		https://twitter.com/ArtidoroPagnoni/status/1661438274894966784

Sends response to the user

Summary -
Step1: Analyze user query - Tool: No tool
Step2: Search Arxiv for QLoRA - Tool: arxiv
Step3: Extract author names - Tool: No tool
Step4: Search for latest tweets - Tool: tavily_search_results_json
Step5: Format and return response - Tool: No tool (text formatting internally)

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [16]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["question"])]}

def parse_output(input_state):
  return input_state["messages"][-1].content

agent_chain = convert_inputs | compiled_graph | parse_output

In [17]:
agent_chain.invoke({"question" : "What is RAG?"})

"RAG stands for Retrieval-Augmented Generation. It is a technique used in natural language processing (NLP) that combines retrieval-based methods with generative models to improve the quality and relevance of generated text. Here's a brief overview of how it works:\n\n1. **Retrieval**: In the first step, relevant documents or pieces of information are retrieved from a large corpus or database. This is typically done using a retrieval model that identifies the most relevant documents based on the input query or context.\n\n2. **Augmentation**: The retrieved documents are then used to augment the input to a generative model. This means that the generative model has access to additional context or information that can help it produce more accurate and contextually relevant responses.\n\n3. **Generation**: Finally, a generative model, such as a transformer-based language model, uses the augmented input to generate a response or piece of text. The presence of additional context from the ret

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]
```

####🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions.

In [19]:
questions = [
    "What is the best recipe for a chocolate cake?",
    "What is the capital of France?",
    "What is the meaning of life?",
    "What is the difference between a dog and a cat?",
    "Why is the sky blue?",
    "What are the ingredients for coq au vin?",
]

answers = [
    {"must_mention" : ["chocolate", "cake"]},
    {"must_mention" : ["Paris", "France"]},
    {"must_mention" : ["42", "meaning"]},
    {"must_mention" : ["dog", "cat"]},
    {"must_mention" : ["blue", "sky"]},
    {"must_mention" : ["coq", "vin"]},
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [20]:
from langsmith import Client

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the QLoRA Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=[{"question" : q} for q in questions],
    outputs=answers,
    dataset_id=dataset.id,
)

#### ❓ Question #3:

How are the correct answers associated with the questions?
LSY ANSWER: using positional indexing in the questions and answers lists. Specifically:
	1.	Each question in questions corresponds to the answer at the same index in answers.
	2.	The client.create_examples() function takes:
	•	inputs=[{"question": q} for q in questions] → Creates a list of dictionaries, each containing a question.
	•	outputs=answers → Assigns the corresponding answer to each question using their positions.
	•	Since inputs and outputs are passed in order, the first question gets the first answer, the second question gets the second answer, and so on.

> NOTE: Feel free to indicate if this is problematic or not
Philosophical Response:
Defining correctness through predefined “must-mention” words assumes that truth is rigid and absolute, but meaning is often fluid, contextual, and evolving. If the chosen words are incorrect or incomplete, the system enforces a false standard of correctness, dismissing valid responses that capture the essence of an answer in different terms. 

Technical Response:
If the “must-mention” words are wrong, incomplete, or overly strict, the evaluation system will misclassify correct answers as incorrect, leading to false negatives. For example, if “Paris” is required for the answer to “What is the capital of France?” but the response says “The capital is a major European city known for the Eiffel Tower,” it would be unfairly marked incorrect.

### Task 2: Adding Evaluators

Now we can add a custom evaluator to see if our responses contain the expected information.

We'll be using a fairly naive exact-match process to determine if our response contains specific strings.

In [21]:
from langsmith.evaluation import EvaluationResult, run_evaluator

@run_evaluator
def must_mention(run, example) -> EvaluationResult:
    prediction = run.outputs.get("output") or ""
    required = example.outputs.get("must_mention") or []
    score = all(phrase in prediction for phrase in required)
    return EvaluationResult(key="must_mention", score=score)

#### ❓ Question #4:

What are some ways you could improve this metric as-is?
LSY ANSWER:
1.	Use semantic similarity scoring instead of exact word matching.
2.	Allow for variations in phrasing or structure.
3.	Include more context-specific keywords.
4.	Use embedding models to compare the overall meaning rather than enforcing rigid keyword inclusion. 
5.  Use fuzzy matching to detect words even if they appear in a different order.



> NOTE: Alternatively you can suggest where gaps exist in this method.

Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [22]:
experiment_results = client.evaluate(
    agent_chain,
    data=dataset_name,
    evaluators=[must_mention],
    experiment_prefix=f"RAG Pipeline - Evaluation - {uuid4().hex[0:4]}",
    metadata={"version": "1.0.0"},
)

/opt/homebrew/Caskroom/miniforge/base/envs/jupyter-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'RAG Pipeline - Evaluation - 37b4-b2cb0c63' at:
https://smith.langchain.com/o/fb0d5d7c-8599-4482-8163-c1156c9773c9/datasets/1aae2f00-f9d3-45f9-a23d-1d33ff3588d6/compare?selectedSessions=8cbd6a49-d63a-42ca-ac06-6c664d30ceca




6it [00:36,  6.07s/it]


In [23]:
experiment_results

<ExperimentResults RAG Pipeline - Evaluation - 37b4-b2cb0c63>

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [24]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

####🏗️ Activity #5:

Please write markdown for the following cells to explain what each is doing.

##### YOUR MARKDOWN HERE
LSY ANSWER:

## Creating a Graph with a Helpfulness Check

The following code initializes a **StateGraph** using `AgentState` and adds two nodes:
- **"agent"**: Calls the model (`call_model`).
- **"action"**: Executes a tool (`tool_node`).

In [25]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### YOUR MARKDOWN HERE
LSY ANSWER: 

## Setting the Entry Point for the Graph

After defining the **StateGraph** and adding nodes, we need to specify the **entry point**—the node where execution begins. In this case, we set `"agent"` as the starting node.

In [26]:
graph_with_helpfulness_check.set_entry_point("agent")

##### YOUR MARKDOWN HERE

LSY ANSWER:
## Implementing a Helpfulness Check in LangChain

The following code defines a function, `tool_call_or_helpful`, that determines the next step in a conversation-based workflow. It evaluates whether the response is helpful, requires a tool call, or should continue.

### **Function Logic**
1. **Check for Tool Calls**  
   - If the last message contains a tool call, return `"action"`, directing the workflow to a tool execution step.

2. **Retrieve Initial and Final Messages**  
   - The function extracts the **first** and **last** messages from the conversation history.

3. **Limit Conversation Length**  
   - If the conversation exceeds **10 messages**, return `"END"` to stop execution.

4. **Evaluate Helpfulness**  
   - A `PromptTemplate` asks whether the final response is **helpful**.
   - The **GPT-4 model** evaluates the response and returns `"Y"` (helpful) or `"N"` (not helpful).
   - If `"Y"`, return `"end"`, otherwise return `"continue"`.

Function tool_call_or_helpful(state):
    # Check if last message contains tool calls
    IF last_message has tool_calls:
        RETURN "action"
    
    # Get initial query and final response from state
    initial_query = first message in state
    final_response = last message in state
    
    # Check if we've exceeded maximum iterations
    IF number of messages > 10:
        RETURN "END"
    
    # Create prompt to check helpfulness
    prompt = create template asking if response is helpful (Y/N)
    
    # Set up helpfulness checking pipeline
    model = initialize GPT-4 model
    chain = combine prompt + model + string parser
    
    # Check helpfulness
    helpfulness_result = run chain with initial query and final response
    
    IF response contains "Y":
        RETURN "end"      # Response is helpful, end the conversation
    ELSE:
        RETURN "continue" # Response needs improvement, continue the conversation

This function acts as a router that:
1. Checks if tools need to be called
2. Enforces a maximum iteration limit
3. Evaluates response helpfulness
4. Determines whether to continue improving the response or end the conversation

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4")

  helpfulness_chain = prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

####🏗️ Activity #4:

Please write what is happening in our `tool_call_or_helpful` function!

##### YOUR MARKDOWN HERE
LSY ANSWER:

The function `tool_call_or_helpful` is used to **determine the next step in a conversational workflow** by analyzing the latest state of the conversation. It decides whether to:
1. **Call a tool** (`"action"`)
2. **Continue the conversation** (`"continue"`)
3. **End the conversation** (`"end"`)

### **How It Works**
- The function is used in `graph_with_helpfulness_check.add_conditional_edges()` to define **conditional transitions** from the `"agent"` node.
- Based on the function's return value, the flow moves to:
  - `"action"` if a tool call is needed.
  - `"agent"` if the conversation should continue.
  - `"end"` if the response is helpful enough to stop.

In [29]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### YOUR MARKDOWN HERE
LSY ANSWER:

## Adding an Edge from Action to Agent

The following code creates a directed edge in the **StateGraph**, allowing execution to transition from the `"action"` node back to the `"agent"` node.

In [28]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### YOUR MARKDOWN HERE
LSY ANSWER:

## Compiling the Graph with Helpfulness Check

The following code compiles the **StateGraph**, finalizing its structure and preparing it for execution.

In [29]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### YOUR MARKDOWN HERE
LSY ANSWER:

## Streaming Responses with Helpfulness Check

The following code initializes a conversation input and streams updates from the compiled graph.

In [30]:
inputs = {"messages" : [HumanMessage(content="Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_l2EcQcAuntJJPEl7syjmDQ1k', 'function': {'arguments': '{"query": "LoRA in machine learning"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_TWrgdmmQXW5XE3vzaPll7P7w', 'function': {'arguments': '{"query": "Tim Dettmers"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_01zEcpZ9bpssGmUFYGBYRtBa', 'function': {'arguments': '{"query": "Attention in machine learning"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 177, 'total_tokens': 258, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4691090a87', 'finish_rea

### Task 4: LangGraph for the "Patterns" of GenAI

Let's ask our system about the 4 patterns of Generative AI:

1. Prompt Engineering
2. RAG
3. Fine-tuning
4. Agents

In [31]:
patterns = ["prompt engineering", "RAG", "fine-tuning", "LLM-based agents"]

In [33]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")









Fine-tuning is a process in machine learning where a pre-trained model is further trained on a smaller, task-specific dataset. This allows the model to adapt to specific tasks or domains while leveraging the general knowledge it has already acquired during its initial training on a larger dataset. Fine-tuning is particularly useful because it can significantly reduce the amount of data and computational resources required to train a model for a specific task, as the model starts with a strong foundation of general knowledge.

Fine-tuning became prominent with the rise of transfer learning techniques, especially in the context of deep learning and natural language processing (NLP). One of the key moments when fine-tuning gained significant attention was with the introduction of models like BERT (Bidirectional Encoder Representations from Transformers) by Google in 2018. BERT demonstrated that fine-tuning a pre-trained language model on specific tasks could achieve state-of-the-a